# Fine-mapping 
## Description

This notebook fine-maps cis-regions on the toy example dataset using three
complementary methods from the xQTL protocol:

- **Stage 1 — SuSiE TWAS** (`mnm_regression.ipynb susie_twas`): individual-level
  fine-mapping + TWAS weights, using **in-sample LD** computed directly from the
  study genotypes.
- **Stage 2 — RSS (summary-statistics fine-mapping)** (`rss_analysis.ipynb univariate_rss`):
  fine-maps a GWAS region using an **external LD reference (LD sketch)**.
- **Stage 3 — fSuSiE** (`mnm_regression.ipynb fsusie`): functional SuSiE
  fine-mapping, also using in-sample LD.

It consumes the genotype, phenotype and covariate outputs produced by the
upstream user-friendly notebooks: `2_genotype_preprocessing.ipynb`,
`3_genotype_pca.ipynb`, `4_covariates_preprocessing.ipynb`,
and the per-chromosome cis phenotype list from `1_phenotype_preprocessing.ipynb`.


## A note on LD references in this toy example

`susie_twas` can use **either** in-sample LD **or** the external LD sketch. By default it uses **in-sample LD** (computed from the study genotypes), but if you pass `--ld_reference_meta_file` its fine-mapping will instead load LD from that sketch (see Step 1). `fsusie` only uses **in-sample LD**; it has no external-LD option.

`univariate_rss` works from GWAS **summary statistics** and therefore *requires* an external LD reference. This toy example ships a small, **self-contained LD sketch** built from the same 60-sample genotype with `pipeline/rss_ld_sketch.ipynb`. It lives under `input/ld_reference/` and is referenced through `input/ld_reference/protocol_example.ld_meta_file.tsv`.

The sketch covers **all autosomes (chr1-chr22)** as per-chromosome PLINK2 **pgen** triplets (`.pgen`/`.pvar`/`.psam`, plus a `.afreq`), one block directory per chromosome. The meta file has columns `#chr start end path`, where `path` is the pgen **prefix** (no extension) resolved relative to the meta file own directory. `pecotmr` `load_LD_matrix` auto-detects the pgen format and reconstructs the LD on the fly. In a full analysis you would point `--ld-meta-data` at a population-scale reference (e.g. ADSP EUR); the sketch demonstrates the identical interface on a small, portable file.


## Input Files

| File | Produced by | Used as |
|------|-------------|---------|
| `output/genotype_by_chrom/protocol_example.genotype.merged.plink_qc.genotype_by_chrom_files.txt` | `2_genotype_preprocessing.ipynb` | `--genoFile` (per-chrom list) |
| `output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt` | `1_phenotype_preprocessing.ipynb` | `--phenoFile` |
| `output/covariate/*.Marchenko_PC.gz` | `4_covariates_preprocessing.ipynb` | `--covFile` |
| `reference_data/TAD/TADB_enhanced_cis.bed` | reference data | `--customized-association-windows` |
| `input/ld_reference/protocol_example.ld_meta_file.tsv` | `rss_ld_sketch.ipynb` (LD sketch) | `--ld-meta-data` (RSS) |
| `data/gwas_meta_data.txt` | reference GWAS sumstats | `--gwas-meta-data` (RSS) |


## Steps

### **Step 1.** SuSiE TWAS fine-mapping (in-sample LD or LD sketch)

Runs `mnm_regression.ipynb susie_twas` on a single example gene (`ENSG00000130538`, chr22).

The optional `--ld_reference_meta_file` controls the LD used by **both** the fine-mapping and the TWAS-weights stages. When you pass it (as below, pointing at the bundled all-chromosome LD sketch `input/ld_reference/protocol_example.ld_meta_file.tsv`), fine-mapping loads LD from that sketch via `pecotmr load_LD_matrix`. When you omit it, the step falls back to **in-sample LD** computed from the study genotypes themselves. The default in this example uses the sketch so both stages share one consistent LD source.

This produces a fine-mapping result (`*.univariate_bvsr.rds`) and TWAS weights (`*.univariate_twas_weights.rds`).


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
head output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt

### **Step 2.** RSS fine-mapping (summary statistics + LD sketch)

Runs `rss_analysis.ipynb univariate_rss` on a GWAS region
(`22:49355984-50799822`) that overlaps the LD sketch. The LD reference is
supplied through `input/ld_reference/protocol_example.ld_meta_file.tsv`; the GWAS summary
statistics are described in `data/gwas_meta_data.txt`. This produces
`*.univariate_susie_rss.rds`.


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
mkdir output/mnm_regression/susie_twas
mkdir output/mnm_regression/susie_twas/data
sos run pipeline/mnm_regression.ipynb susie_twas \
    --name test_susie_twas \
    --genoFile output/genotype_by_chrom/wgs.merged.plink_qc.1.bed \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --phenotype-names test_pheno \
    --max-cv-variants 5000 --ld_reference_meta_file reference_data/ADSP_R4_EUR/ld_meta_file.tsv \
    --region-name ENSG00000049246 ENSG00000054116 ENSG00000116678 \
    --save-data \
    --cwd output/mnm_regression/susie_twas

### **Step 3.** fSuSiE fine-mapping (in-sample LD)

Runs `mnm_regression.ipynb fsusie` (functional SuSiE) on the same example gene
(`ENSG00000130538`). Like `susie_twas`, it uses in-sample LD and needs no
external LD reference. This produces `*.fsusie_mixture_normal_TI__top_pc_weights.rds`.


In [ ]:
# change to R code
setwd('/home/ubuntu/xqtl_protocol_exercise')
uv_susie = readRDS('output/mnm_regression/susie_twas/fine_mapping/test_susie_twas.chr1_ENSG00000116678.univariate_bvsr.rds')
names(uv_susie)
str(uv_susie)

## Output Files

| File | Stage |
|------|-------|
| `output/finemapping/susie_twas/fine_mapping/protocol_example_susie_twas.chr22_ENSG00000130538.univariate_bvsr.rds` | Step 1 |
| `output/finemapping/susie_twas/twas_weights/protocol_example_susie_twas.chr22_ENSG00000130538.univariate_twas_weights.rds` | Step 1 |
| `output/finemapping/rss_analysis/univariate_rss/RSS_QC_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds` | Step 2 |
| `output/finemapping/fsusie/fsus/protocol_example_fsusie.chr22_15528191_15529138.fsusie_mixture_normal_TI__top_pc_weights.rds` | Step 3 |


## Anticipated Results

Each stage writes an `.rds` result object for the analyzed region. On this
minimal toy dataset the credible sets and posterior inclusion probabilities are
not biologically meaningful; the goal is to confirm that all three fine-mapping
pipelines run end-to-end and emit their expected output objects.


In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
head data/mnm_regression/gwas_meta_data.txt

study_id	chrom	file_path	column_mapping_file	n_sample	n_case	n_control
AD_Bellenguez_2022	0	data/twas/AD_Bellenguez_2022_RSS_QC_RAISS_imputed.tsv.gz	data/twas/Bellenguez.yml	0	111326	677663


In [ ]:
sos run pipeline/rss_analysis.ipynb univariate_rss \
    --ld-meta-data reference_data/ADSP_R4_EUR/ld_meta_file.tsv \
    --gwas-meta-data data/mnm_regression/gwas_meta_data.txt \
    --qc_method "rss_qc" --impute \
    --finemapping_method "susie_rss" \
    --cwd output/rss_analysis \
    --skip_analysis_pip_cutoff 0 \
    --skip_regions 6:25000000-35000000 \
    --region_name 22:49355984-50799822

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Running get_analysis_regions: 
INFO: Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO: NumExpr defaulting to 16 threads.
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: Running univariate_rss: 
INFO: univariate_rss is completed.
INFO: univariate_rss output:   /mnt/vast/hpc/homes/al4225/xqtl_protocol_data/output/rss_analysis/univariate_rss/RSS_QC_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds
INFO: Workflow univariate_rss (ID=w60aca78e2357ae7c) is executed successfully with 2 completed steps.


In [ ]:
# R code
rss = readRDS('/mnt/vast/hpc/homes/al4225/xqtl_protocol_data/output/rss_analysis/univariate_rss/RSS_QC_RAISS_imputed.chr22_49355984_50799822.univariate_susie_rss.rds')
names(rss)
str(rss)

[1] "chr22_49355984_50799822"

List of 1
 $ chr22_49355984_50799822:List of 1
  ..$ AD_Bellenguez_2022:List of 2
  .. ..$ susie_rss_RSS_QC_RAISS_imputed:List of 5
  .. .. ..$ variant_names       : chr [1:7610] "22:49356357:G:A" "22:49356408:G:A" "22:49356690:C:T" "22:49357190:G:A" ...
  .. .. ..$ analysis_script     : chr "library(pecotmr)\nlibrary(dplyr)\nlibrary(data.table)\nskip_region = c(\"6:25000000-35000000\")\nstudies = c(\""| __truncated__
  .. .. ..$ sumstats            :List of 1
  .. .. .. ..$ z: num [1:7610] 0.3034 1.2604 0.0707 1.2353 -1.2175 ...
  .. .. ..$ susie_result_trimmed:List of 9
  .. .. .. ..$ pip           : num [1:7610] 0.000471 0.000674 0.000461 0.000654 0.000659 ...
  .. .. .. ..$ sets          :List of 3
  .. .. .. .. ..$ cs                : NULL
  .. .. .. .. ..$ coverage          : NULL
  .. .. .. .. ..$ requested_coverage: num 0.95
  .. .. .. ..$ cs_corr       : logi NA
  .. .. .. ..$ sets_secondary:List of 2
  .. .. .. .. ..$ coverage_0.7:List of 2
  .. .. .. .. .. ..$ sets   :List o

In [ ]:
cd /home/ubuntu/xqtl_protocol_exercise
sos run pipeline/mnm_regression.ipynb fsusie \
    --cwd output/fsusie/ \
    --name test_fsusie \
    --genoFile output/genotype_by_chrom/wgs.merged.plink_qc.genotype_by_chrom_files.txt \
    --phenoFile output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.region_list.txt \
    --covFile output/covariate/bulk_rnaseq_tmp_matrix.low_expression_filtered.outlier_removed.tmm.expression.covariates.wgs.merged.plink_qc.plink_qc.prune.pca.Marchenko_PC.gz \
    --numThreads 8 \
    --customized-association-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --save-data \
    --region-name ENSG00000049246 ENSG00000054116 ENSG00000116678 

/home/al4225/.pixi/envs/python/lib/python3.12/site-packages/sos/targets.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
INFO: Note: NumExpr detected 32 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 16.
INFO: NumExpr defaulting to 16 threads.
INFO: Running get_analysis_regions: 
Loading customized association analysis window from reference_data/TAD/TADB_enhanced_cis.bed
INFO: get_analysis_regions is completed.
INFO: get_analysis_regions output:   regional_data
INFO: Running fsusie: 
INFO: fsusie (index=1) is completed.
INFO: fsusie (index=0) is completed.
INFO: fsusie (index=2) is completed.
INFO: fsusie output:   /mnt/vast/hpc/homes/al4225/xqtl_protocol_data/output/fsusie/fsus/test_fsusie.chr1_7784319_7845176.fsusie_mixture_normal_TI__top_pc_we